# Lab 2: Vectorless RAG — Advanced Scenarios

## Setup

### Install Dependencies

In [ ]:
!pip install -q pageindex openai python-dotenv pymupdf

### Import Libraries

In [ ]:
import os
import json
import re
import pymupdf
from openai import OpenAI
from dotenv import load_dotenv

### Load API Keys

In [ ]:
load_dotenv("../.env")

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# If keys are missing, prompt the user to enter them
if not PAGEINDEX_API_KEY:
    PAGEINDEX_API_KEY = input("Enter your PageIndex API key (get one at https://pageindex.ai): ").strip()
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("Enter your OpenRouter API key (get one at https://openrouter.ai): ").strip()

print("Keys loaded.")

### Set up the LLM

In [ ]:
def call_llm(prompt, model="meta-llama/llama-4-scout-17b-16e-instruct"):
    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)
    return client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}], temperature=0, max_tokens=1024
    ).choices[0].message.content.strip()

---
## Load and Parse the PDF

### Define PDF Path

In [ ]:
PDF_PATH = "data/CCS 3.31.25 Earnings Release 8-K Exhibit 99.1.pdf"

### Extract Text from PDF

In [ ]:
doc = pymupdf.open(PDF_PATH)
page_texts = {i+1: doc.load_page(i).get_text() for i in range(len(doc))}
doc.close()
print(f"Extracted text from {len(page_texts)} pages.")

---
## Build Document Tree (with caching)

### Set Up Caching and PageIndex

In [ ]:
from pageindex import PageIndexClient
from pageindex import utils
import time

CACHE_PATH = f"cache/{os.path.basename(PDF_PATH).replace('.pdf', '_tree.json')}"
os.makedirs("cache", exist_ok=True)

### Load or Build the Tree

In [ ]:
tree = None
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH) as f:
        tree = json.load(f)
    print("Loaded tree from cache.")

if tree is None:
    pi = PageIndexClient(api_key=PAGEINDEX_API_KEY)
    result = pi.submit_document(PDF_PATH)
    doc_id = result["doc_id"]
    print(f"Submitted: {doc_id}")

    elapsed = 0
    while elapsed < 300:
        if pi.is_retrieval_ready(doc_id):
            break
        time.sleep(5)
        elapsed += 5
        print(f"  {elapsed}s...")
    else:
        raise TimeoutError("PageIndex timeout")

    tree = pi.get_tree(doc_id, node_summary=True)["result"]
    with open(CACHE_PATH, "w") as f:
        json.dump(tree, f, indent=2)
    print("Tree cached.")

utils.print_tree(tree, exclude_fields=["text"])

---
## Helper Functions

### Parse JSON

In [ ]:
def parse_json(text):
    """Extract and robustly parse JSON from LLM response, ignoring extra text."""
    text = re.sub(r"```json\s*|\s*```", "", text.strip())
    
    # Locate the outermost curly braces
    s = text.find("{")
    
    # Walk backward from the end to find the true matching closing brace for the JSON object
    # This prevents catching trailing text as part of the JSON payload
    e = text.rfind("}")
    
    if s != -1 and e != -1:
        text = text[s:e+1]
    
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    
    # Fallback: extract thinking and node_list with regex when JSON is malformed
    thinking = ""
    m_think = re.search(r'"thinking"\s*:\s*"((?:[^"\\]|\\.)*)"', text, re.DOTALL)
    if m_think:
        thinking = m_think.group(1)
    
    node_list = []
    m_nodes = re.search(r'"node_list"\s*:\s*\[(.*?)\]', text, re.DOTALL)
    if m_nodes:
        node_list = re.findall(r'"(\d+)"', m_nodes.group(1))
    
    return {"thinking": thinking, "node_list": node_list}

### Retrieve Nodes

In [ ]:
def retrieve_nodes(query):
    """Use LLM to find relevant nodes and map the hop-by-hop traversal with titles."""
    tree_slim = utils.remove_fields(tree.copy(), fields=["text"])
    prompt = f"""
Given a question and a document tree, find nodes likely to contain the answer.
You must perform multi-hop reasoning. Document your exact step-by-step traversal path.

Question: {query}

Tree:
{json.dumps(tree_slim, indent=2)}

JSON only:
{{
  "hops": [
    {{"step": 1, "node_id": "id1", "section_title": "Name of the table or section", "reason": "Looked here first because..."}},
    {{"step": 2, "node_id": "id2", "section_title": "Name of the next table", "reason": "Then realized I needed X..."}}
  ],
  "node_list": ["id1", "id2"]
}}
"""
    return parse_json(call_llm(prompt))

### Get Context

In [ ]:
def get_context(node_list):
    """Extract text from pages covered by the given nodes."""
    node_map = utils.create_node_mapping(tree, include_page_ranges=True, max_page=len(page_texts))
    texts, seen = [], set()
    for nid in node_list:
        info = node_map[nid]
        for p in range(info["start_index"], info["end_index"] + 1):
            if p not in seen and p in page_texts:
                texts.append(f"--- Page {p} ---\n{page_texts[p]}")
                seen.add(p)
    return "\n\n".join(texts)

### Plot traversal

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def plot_traversal(hops):
    """Draws a visual graph of the LLM's multi-hop reasoning path."""
    G = nx.DiGraph()
    
    # Define the starting point
    G.add_node("User Query", color="#e3f2fd") # Light blue
    
    node_colors = ["#e3f2fd"]
    pos = {"User Query": (0, 1)}
    
    # Build the path
    for i, hop in enumerate(hops):
        label = f"Hop {hop['step']}:\nNode {hop['node_id']}"
        G.add_node(label, color="#fff3e0") # Light orange
        node_colors.append("#fff3e0")
        
        # Connect to the previous step (or query if it is the first step)
        if i == 0:
            G.add_edge("User Query", label)
        else:
            prev_label = f"Hop {hops[i-1]['step']}:\nNode {hops[i-1]['node_id']}"
            G.add_edge(prev_label, label)
            
        # Layout positioning
        pos[label] = (i + 1, i % 2) # Staggers them slightly for a "hopping" look
        
    # Draw the graph
    plt.figure(figsize=(10, 4))
    nx.draw(G, pos, with_labels=True, node_color=node_colors, 
            node_size=4000, font_size=9, font_weight="bold", 
            edge_color="#616161", arrows=True, arrowsize=20)
    
    plt.title("Vectorless RAG : Reasoning Path", fontsize=14)
    plt.margins(0.2)
    plt.show()

---
# Scenario 1: Multi-Hop Attribute Aggregation

### Define Multi-Hop Query

In [ ]:
# Change this to a question that requires info from multiple sections
QUERY_MULTI_HOP = "What did the Executive Chairman say was the cause of the slower spring selling season, and what is the new updated full-year home delivery guidance range?"

### Retrieve Nodes for Multi-Hop Query

In [ ]:
result = retrieve_nodes(QUERY_MULTI_HOP)

print("--- RAG Traversal Path ---")
for hop in result["hops"]:
    print(f"Hop {hop['step']} (Node {hop['node_id']}): {hop['reason']}")

print("\nFinal Nodes Retrieved:", result["node_list"])

### Answer Multi-Hop Query

In [ ]:
context = get_context(result["node_list"])
answer = call_llm(f"Context:\n{context}\n\nQuestion: {QUERY_MULTI_HOP}\n\nAnswer concisely.")
print("Answer:", answer)

In [ ]:
# Call the function with our results
plot_traversal(result["hops"])

---
# Scenario 2: Structured Data Fidelity

### Define Structured Data Query

In [ ]:
# Change this to a question about extracting specific values from a table
QUERY_STRUCTURED = "According to the Home Deliveries table, what was the exact number of homes delivered and the average sales price for the 'Texas' region in the first quarter of 2025?"

### Retrieve Nodes for Structured Data

In [ ]:
result = retrieve_nodes(QUERY_STRUCTURED)

print("--- Table Analysis Path (Structured Data) ---")
for step in result["hops"]:
    print(f"Step {step['step']} (Table Node {step['node_id']}): {step['reason']}")

print("\nFinal Nodes Retrieved:", result["node_list"])

### Answer Structured Data Query

In [ ]:
context = get_context(result["node_list"])
answer = call_llm(f"Context:\n{context}\n\nQuestion: {QUERY_STRUCTURED}\n\nAnswer concisely.")
print("Answer:", answer)

In [ ]:
# Generate the visual plot for the table extraction
plot_traversal(result["hops"])

---
## Try It Yourself

Change the `QUERY_*` variables and re-run the cells. Here are some generic patterns to try:

In [ ]:
examples = [
    "What was the net income and how does it compare to the prior year period?",
    "What is the number of net new home contracts for the West region?",
    "What were the total homebuilding gross margins and how do they break down by region?",
]

for q in examples:
    result = retrieve_nodes(q)
    print(f"{q[:50]}... -> {len(result['node_list'])} nodes")